# Lab | Hypothesis Testing

**Objective**

Welcome to the Hypothesis Testing Lab, where we embark on an enlightening journey through the realm of statistical decision-making! In this laboratory, we delve into various scenarios, applying the powerful tools of hypothesis testing to scrutinize and interpret data.

From testing the mean of a single sample (One Sample T-Test), to investigating differences between independent groups (Two Sample T-Test), and exploring relationships within dependent samples (Paired Sample T-Test), our exploration knows no bounds. Furthermore, we'll venture into the realm of Analysis of Variance (ANOVA), unraveling the complexities of comparing means across multiple groups.

So, grab your statistical tools, prepare your hypotheses, and let's embark on this fascinating journey of exploration and discovery in the world of hypothesis testing!

**Challenge 1**

In this challenge, we will be working with pokemon data. The data can be found here:

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/pokemon.csv

In [1]:
#libraries
import pandas as pd
import scipy.stats as st
import numpy as np



In [3]:
# Load Pokémon into its own variable to avoid collisions with later datasets
df_poke = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/pokemon.csv")
df_poke.head()

,Name,Type 1,Type 2,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,Bulbasaur,Grass,Poison,45,49,49,65,65,45,1,False
1,Ivysaur,Grass,Poison,60,62,63,80,80,60,1,False
2,Venusaur,Grass,Poison,80,82,83,100,100,80,1,False
3,Mega Venusaur,Grass,Poison,80,100,123,122,120,80,1,False
4,Charmander,Fire,NaN,39,52,43,60,50,65,1,False
...,...,...,...,...,...,...,...,...,...,...,...
795,Diancie,Rock,Fairy,50,100,150,100,150,50,6,True
796,Mega Diancie,Rock,Fairy,50,160,110,160,110,110,6,True
797,Hoopa Confined,Psychic,Ghost,80,110,60,150,130,70,6,True
798,Hoopa Unbound,Psychic,Dark,80,160,60,170,130,80,6,True


- We posit that Pokemons of type Dragon have, on average, more HP stats than Grass. Choose the propper test and, with 5% significance, comment your findings.

In [1]:
# Hypothesis: Dragon-type Pokémon have, on average, higher HP than non-Dragon Pokémon.
# H0: μ_Dragon ≤ μ_Others   vs   H1: μ_Dragon > μ_Others

# Identify Dragon Pokémon if 'Dragon' appears in either Type 1 or Type 2
is_dragon = df_poke['Type 1'].str.contains('Dragon', case=False, na=False)
if 'Type 2' in df_poke.columns:
    is_dragon = is_dragon | df_poke['Type 2'].str.contains('Dragon', case=False, na=False)

dragon_hp = df_poke.loc[is_dragon, 'HP'].dropna()
other_hp  = df_poke.loc[~is_dragon, 'HP'].dropna()

# Welch's t-test (independent samples, unequal variances). We'll convert from two-sided to one-sided.
t_stat, p_two_sided = st.ttest_ind(dragon_hp, other_hp, equal_var=False, nan_policy='omit')

# One-sided p-value (H1: Dragon > Others)
p_one_sided = p_two_sided/2 if t_stat > 0 else 1 - p_two_sided/2

alpha = 0.05
print(f"n_dragon={len(dragon_hp)}, n_others={len(other_hp)}")
print(f"mean_dragon_HP={dragon_hp.mean():.2f}, mean_others_HP={other_hp.mean():.2f}")
print(f"Welch t={t_stat:.3f}, one-sided p={p_one_sided:.4g}")
print("Decision (α=0.05):", "Reject H0 (Dragons have higher HP)" if p_one_sided < alpha else "Fail to reject H0")

- We posit that Legendary Pokemons have different stats (HP, Attack, Defense, Sp.Atk, Sp.Def, Speed) when comparing with Non-Legendary. Choose the propper test and, with 5% significance, comment your findings.


In [18]:
# Hypothesis: Legendary Pokémon have different stats than non-Legendary (two-sided tests).
# We'll run Welch's t-tests for each stat and show Bonferroni-adjusted p-values.

# Find the Legendary column name robustly (accepts 'Legendary', 'is_legendary', etc.)
legend_col = None
for c in df_poke.columns:
    norm = c.lower().replace(" ", "").replace(".", "").replace("_", "").replace("?", "")
    if norm in ("legendary", "islegendary"):
        legend_col = c
        break
if legend_col is None:
    raise KeyError("Legendary / is_legendary column not found in Pokémon dataset.")

# Make a boolean mask for legendary
is_legendary = df_poke[legend_col]
if is_legendary.dtype != bool:
    is_legendary = is_legendary.astype(str).str.lower().isin(['true','1','yes','t','legendary'])

# Stats columns (keep only those present)
stats_cols = [c for c in ['HP','Attack','Defense','Sp. Atk','Sp. Def','Speed'] if c in df_poke.columns]

results = []
for col in stats_cols:
    x = df_poke.loc[is_legendary, col].dropna()
    y = df_poke.loc[~is_legendary, col].dropna()
    t_stat, p_two = st.ttest_ind(x, y, equal_var=False, nan_policy='omit')
    results.append({
        'stat': col,
        'n_legendary': len(x),
        'n_nonlegendary': len(y),
        'mean_legendary': float(x.mean()) if len(x) else np.nan,
        'mean_nonlegendary': float(y.mean()) if len(y) else np.nan,
        't': float(t_stat),
        'p_two_sided': float(p_two)
    })

res_df = pd.DataFrame(results)
m = len(res_df)
res_df['p_bonferroni'] = (res_df['p_two_sided'] * m).clip(upper=1.0)
res_df.sort_values('p_two_sided', inplace=True)
res_df.reset_index(drop=True, inplace=True)
res_df

**Challenge 2**

In this challenge, we will be working with california-housing data. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/california_housing.csv

In [5]:
df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/california_housing.csv")
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-114.31,34.19,15.0,5612.0,1283.0,1015.0,472.0,1.4936,66900.0
1,-114.47,34.40,19.0,7650.0,1901.0,1129.0,463.0,1.8200,80100.0
2,-114.56,33.69,17.0,720.0,174.0,333.0,117.0,1.6509,85700.0
3,-114.57,33.64,14.0,1501.0,337.0,515.0,226.0,3.1917,73400.0
4,-114.57,33.57,20.0,1454.0,326.0,624.0,262.0,1.9250,65500.0


**We posit that houses close to either a school or a hospital are more expensive.**

- School coordinates (-118, 34)
- Hospital coordinates (-122, 37)

We consider a house (neighborhood) to be close to a school or hospital if the distance is lower than 0.50.

Hint:
- Write a function to calculate euclidean distance from each house (neighborhood) to the school and to the hospital.
- Divide your dataset into houses close and far from either a hospital or school.
- Choose the propper test and, with 5% significance, comment your findings.
 

In [ ]:
# Compute Euclidean distances to the given school and hospital, then flag 'near' vs 'far'
import numpy as np

def euclidean_distance(lon, lat, lon0, lat0):
    return np.sqrt((lon - lon0)**2 + (lat - lat0)**2)

school = (-118.0, 34.0)
hospital = (-122.0, 37.0)

df['dist_school'] = euclidean_distance(df['longitude'], df['latitude'], school[0], school[1])
df['dist_hospital'] = euclidean_distance(df['longitude'], df['latitude'], hospital[0], hospital[1])
df['dist_min'] = df[['dist_school','dist_hospital']].min(axis=1)

threshold = 0.50  # as specified
df['near'] = df['dist_min'] < threshold

df[['longitude','latitude','dist_school','dist_hospital','dist_min','near']].head()

In [ ]:
# Hypothesis: Houses near (within 0.50 of a school or hospital) are more expensive.
# H0: μ_near ≤ μ_far   vs   H1: μ_near > μ_far  (one-sided)

near_vals = df.loc[df['near'], 'median_house_value'].dropna()
far_vals  = df.loc[~df['near'], 'median_house_value'].dropna()

t_stat, p_two_sided = st.ttest_ind(near_vals, far_vals, equal_var=False, nan_policy='omit')
p_one_sided = p_two_sided/2 if t_stat > 0 else 1 - p_two_sided/2

alpha = 0.05
print(f"n_near={len(near_vals)}, n_far={len(far_vals)}")
print(f"mean_near={near_vals.mean():.2f}, mean_far={far_vals.mean():.2f}")
print(f"Welch t={t_stat:.3f}, one-sided p={p_one_sided:.4g}")
print("Decision (α=0.05):", "Reject H0 (near are pricier)" if p_one_sided < alpha else "Fail to reject H0")